In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

n = 1000
p = 950
sigma_x = 0.1
alpha = 0.05
np.random.seed(2137)

X = np.random.normal(0, sigma_x, size=(n, p))
beta_true = np.zeros(p)
beta_true[:5] = 3

epsilon = np.random.normal(0, 1, size=n)
Y = X @ beta_true + epsilon

k_values = [1, 2, 5, 10, 50, 100, 500, 950]

precalc_b = {}
for k in k_values:
    X_k = X[:, :k]
    xtx = X_k.T @ X_k
    xtx_inv = np.linalg.inv(xtx)
    proj = xtx_inv @ X_k.T 
    precalc_b[k] = {
        'X_k': X_k,
        'xtx_inv': xtx_inv,
        'proj': proj,
        'diag_inv': np.diag(xtx_inv)
    }

In [ ]:
results_b = []

for k in k_values:
    p_b = precalc_b[k]
    X_k = p_b['X_k']
    
    b_hat = p_b['proj'] @ Y
    
    residuals = Y - X_k @ b_hat
    sse = np.sum(residuals**2)
    
    y_hat = X_k @ b_hat
    y_expected = X[:, :k] @ beta_true[:k]
    mse = np.sum((y_hat - y_expected)**2)
    
    aic = n * np.log(sse / n) + 2 * k
    
    sigma2_hat = sse / (n - k)
    se = np.sqrt(sigma2_hat * p_b['diag_inv'])
    t_stats = np.abs(b_hat / se)
    p_values = 2 * (1 - stats.t.cdf(t_stats, df=n-k))
    
    p_val_1 = p_values[0] if k >= 1 else None
    p_val_2 = p_values[1] if k >= 2 else None
    
    significant_indices = np.where(p_values < alpha)[0]
    false_discoveries = np.sum(significant_indices >= 5)
    
    results_b.append({
        'k': k, 'SSE': sse, 'MSE': mse, 'AIC': aic,
        'p-val X1': p_val_1, 'p-val X2': p_val_2, 'FP': false_discoveries
    })

df_b = pd.DataFrame(results_b)
print("Wyniki punktu B:")
print(df_b)

Wyniki punktu B:
     k          SSE         MSE          AIC      p-val X1      p-val X2  FP
0    1  1361.005424    0.335989   310.223709  0.000000e+00           NaN   0
1    2  1282.292165    0.822302   252.649230  0.000000e+00  1.265654e-14   0
2    5   975.340712    7.303343   -14.968421  0.000000e+00  0.000000e+00   0
3   10   971.957965   10.686091    -8.442722  0.000000e+00  0.000000e+00   0
4   50   937.874918   44.769137    35.861312  0.000000e+00  0.000000e+00   0
5  100   873.225175  109.418881    64.438176  2.220446e-16  2.220446e-16   6
6  500   489.259546  493.384509   285.137839  1.572715e-09  2.937206e-11  28
7  950    43.828213  938.815843 -1227.477542  2.950877e-01  1.696306e-01  18


In [ ]:
b_hat_full = precalc_b[950]['proj'] @ Y
sorted_indices = np.argsort(np.abs(b_hat_full))[::-1]

results_c = []

for k in k_values:
    best_k_indices = sorted_indices[:k]
    X_k_c = X[:, best_k_indices]
    
    xtx_c = X_k_c.T @ X_k_c
    b_hat_c = np.linalg.solve(xtx_c, X_k_c.T @ Y)
    
    y_hat_c = X_k_c @ b_hat_c
    sse_c = np.sum((Y - y_hat_c)**2)
    mse_c = np.sum((y_hat_c - (X @ beta_true))**2)
    
    aic_c = n * np.log(sse_c / n) + 2 * k
    
    sigma2_c = sse_c / (n - k)
    se_c = np.sqrt(sigma2_c * np.diag(np.linalg.inv(xtx_c)))
    p_values_c = 2 * (1 - stats.t.cdf(np.abs(b_hat_c / se_c), df=n-k))
    
    sig_mask = p_values_c < alpha
    actual_sig = best_k_indices[sig_mask]
    
    results_c.append({
        'k': k, 'SSE': sse_c, 'MSE': mse_c, 'AIC': aic_c,
        'TP': np.sum(actual_sig < 5), 'FP': np.sum(actual_sig >= 5)
    })

df_c = pd.DataFrame(results_c)
print("Wyniki punktu C:")
print(df_c)

Wyniki punktu C:
     k          SSE         MSE          AIC  TP   FP
0    1  1395.181075  370.429036   335.024210   1    0
1    2  1240.217552  281.272215   219.286810   2    0
2    5  1238.811895  283.767110   224.152771   2    0
3   10  1212.913651  298.996635   213.025441   2    1
4   50  1075.323352  232.215020   172.621409   3    5
5  100   932.999829  209.887729   130.649739   4   13
6  500   263.810763  718.833292  -332.523239   5  332
7  950    43.828213  938.815843 -1227.477542   2   18


In [ ]:
n_sim = 1000
counts_b = {k: {'tp1': 0, 'tp2': 0, 'fp_sum': 0} for k in k_values}
counts_c = {k: {'tp1': 0, 'tp2': 0, 'fp_sum': 0} for k in k_values}
aic_selected_k_b, aic_selected_k_c = [], []

print(f"Rozpoczynam symulację {n_sim} iteracji...")

for i in range(n_sim):
    Y_sim = X @ beta_true + np.random.normal(0, 1, size=n)
    iter_aic_b, iter_aic_c = {}, {}
    
    for k in k_values:
        p_b = precalc_b[k]
        b_hat = p_b['proj'] @ Y_sim
        rss = np.sum((Y_sim - p_b['X_k'] @ b_hat)**2)
        iter_aic_b[k] = n * np.log(rss / n) + 2 * k
        
        se = np.sqrt((rss / (n - k)) * p_b['diag_inv'])
        p_vals = 2 * (1 - stats.t.cdf(np.abs(b_hat / se), df=n-k))
        
        if k >= 1 and p_vals[0] < alpha: counts_b[k]['tp1'] += 1
        if k >= 2 and p_vals[1] < alpha: counts_b[k]['tp2'] += 1
        counts_b[k]['fp_sum'] += np.sum(p_vals[np.where(beta_true[:k] == 0)[0]] < alpha)

    aic_selected_k_b.append(min(iter_aic_b, key=iter_aic_b.get))

    sorted_idx = np.argsort(np.abs(b_hat))[::-1] 
    for k in k_values:
        idx_c = sorted_idx[:k]
        X_c = X[:, idx_c]
        b_hat_c = np.linalg.solve(X_c.T @ X_c, X_c.T @ Y_sim)
        rss_c = np.sum((Y_sim - X_c @ b_hat_c)**2)
        iter_aic_c[k] = n * np.log(rss_c / n) + 2 * k
        
        se_c = np.sqrt((rss_c / (n - k)) * np.diag(np.linalg.inv(X_c.T @ X_c)))
        p_vals_c = 2 * (1 - stats.t.cdf(np.abs(b_hat_c / se_c), df=n-k))
        
        sig_indices = idx_c[p_vals_c < alpha]
        if 0 in sig_indices: counts_c[k]['tp1'] += 1
        if 1 in sig_indices: counts_c[k]['tp2'] += 1
        counts_c[k]['fp_sum'] += np.sum(sig_indices >= 5)

    aic_selected_k_c.append(min(iter_aic_c, key=iter_aic_c.get))

Rozpoczynam symulację 1000 iteracji...


In [ ]:
final_data = []
for k in k_values:
    final_data.append({
        'k': k,
        'Pow X1 (B)': counts_b[k]['tp1'] / n_sim,
        'Pow X2 (B)': counts_b[k]['tp2'] / n_sim,
        'E(FD) (B)': counts_b[k]['fp_sum'] / n_sim,
        'Pow X1 (C)': counts_c[k]['tp1'] / n_sim,
        'Pow X2 (C)': counts_c[k]['tp2'] / n_sim,
        'E(FD) (C)': counts_c[k]['fp_sum'] / n_sim
    })
df_sim_final = pd.DataFrame(final_data)

def save_simple_tex(df, filename, caption, label):
    df_out = df.copy()
    
    def fmt(x):
        if x is None or (isinstance(x, float) and np.isnan(x)): 
            return "---"
        if isinstance(x, (float, np.floating)):
            if abs(x) < 1e-12: return "0.0000"
            if abs(x) < 0.001 or abs(x) > 1e5: return "{:.2e}".format(x)
            return "{:.4f}".format(x)
        return x
        
    df_out = df_out.apply(lambda col: col.map(fmt))
    
    new_cols = [str(c).replace("p-val", "p").replace("E[FD]", "E(FD)") for c in df_out.columns]
    df_out.columns = new_cols

    with open(filename, 'w') as f:
        latex_str = df_out.to_latex(
            index=False, 
            caption=caption, 
            label=label,
            position="H", 
            column_format="l" + "c"*len(df_out.columns), 
            escape=False
        )
        f.write(latex_str)
    print(f"[OK] Zapisano: {filename}")

save_simple_tex(df_b, "tab_zad2_b.tex", "Wyniki regresji dla punktu b", "tab:zad2_b")
save_simple_tex(df_c, "tab_zad2_c.tex", "Wyniki regresji dla punktu c", "tab:zad2_c")
save_simple_tex(df_sim_final, "tab_zad2_d.tex", "Wyniki symulacji (punkt d)", "tab:zad2_d")

[OK] Zapisano: tab_zad2_b.tex
[OK] Zapisano: tab_zad2_c.tex
[OK] Zapisano: tab_zad2_d.tex
